In [1]:
import os
import torch
import torchaudio
import librosa
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [2]:
torchaudio.datasets.SPEECHCOMMANDS(root="./data", download=True)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [4]:
def extract_mfcc(path, max_len=100):
    y, sr = librosa.load(path, sr=16000)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    if mfcc.shape[1] < max_len:
        pad = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0, 0), (0, pad)))
    else:
        mfcc = mfcc[:, :max_len]
    return mfcc.T

In [5]:
class SpeechDataset(Dataset):
    def __init__(self, root, cache_dir="mfcc_cache"):
        cache_path = os.path.join(root, cache_dir)
        mfcc_path = os.path.join(cache_path, "mfccs.dat")
        label_path = os.path.join(cache_path, "labels.npy")
        meta_path = os.path.join(cache_path, "meta.npy")

        assert os.path.exists(mfcc_path), (
            f"Cache not found at {cache_path}. Run `python precompute_mfcc.py` first!"
        )
        shape = tuple(np.load(meta_path))
        self.classes = list(np.load(os.path.join(cache_path, "classes.npy")))
        self.files = list(np.load(os.path.join(cache_path, "files.npy")))

        self.mfccs = np.memmap(mfcc_path, dtype='float32', mode='r', shape=shape)
        self.label_arr = np.load(label_path)

        print(f"Loaded {len(self.files)} cached MFCCs (memory-mapped)")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mfcc = np.array(self.mfccs[idx])
        return torch.tensor(mfcc, dtype=torch.float32), int(self.label_arr[idx])

In [6]:
root = "./data/SpeechCommands/speech_commands_v0.02"

val_list_path = os.path.join(root, "validation_list.txt")
test_list_path = os.path.join(root, "testing_list.txt")

val_files = set()
test_files = set()

if os.path.exists(val_list_path):
    with open(val_list_path) as f:
        val_files = set(line.strip() for line in f)
if os.path.exists(test_list_path):
    with open(test_list_path) as f:
        test_files = set(line.strip() for line in f)

dataset = SpeechDataset(root)
NUM_CLASSES = len(dataset.classes)

train_idx, val_idx, test_idx = [], [], []
for i, filepath in enumerate(dataset.files):
    rel = os.path.relpath(filepath, root).replace("\\", "/")
    if rel in test_files:
        test_idx.append(i)
    elif rel in val_files:
        val_idx.append(i)
    else:
        train_idx.append(i)

train_ds = torch.utils.data.Subset(dataset, train_idx)
val_ds   = torch.utils.data.Subset(dataset, val_idx)
test_ds  = torch.utils.data.Subset(dataset, test_idx)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, num_workers=0, pin_memory=True)

print(f"Classes ({NUM_CLASSES}): {dataset.classes}")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Loaded 105829 cached MFCCs (memory-mapped)
Classes (35): ['backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'follow', 'forward', 'four', 'go', 'happy', 'house', 'learn', 'left', 'marvin', 'nine', 'no', 'off', 'on', 'one', 'right', 'seven', 'sheila', 'six', 'stop', 'three', 'tree', 'two', 'up', 'visual', 'wow', 'yes', 'zero']
Train: 84843, Val: 9981, Test: 11005


In [7]:
class GRUModel(nn.Module):
    def __init__(self, num_classes, input_size=40, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()

        # Pure GRU — raw MFCC frames fed directly, no CNN front-end
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)  # *2 for bidirectional

    def forward(self, x):
        # x: (batch, time_steps, 40) — fed straight into GRU
        _, h = self.gru(x)        # h: (num_layers*2, batch, hidden_size)
        # Take last forward and backward hidden states
        h = torch.cat((h[-2], h[-1]), dim=1)  # (batch, hidden_size*2)
        h = self.dropout(h)
        return self.fc(h)

In [8]:
model = GRUModel(NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(model)

GRUModel(
  (gru): GRU(40, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=35, bias=True)
)


In [9]:
EPOCHS = 15
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for x, y in loop:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = out.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

        loop.set_postfix(loss=loss.item())

    train_acc = correct / total
    scheduler.step()

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            val_correct += (pred == y).sum().item()
            val_total += y.size(0)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}, LR={scheduler.get_last_lr()[0]:.6f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model_gru.pth")
        print(f"  -> Saved best model (Val Acc={val_acc:.4f})")

print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

Epoch 1/15: 100%|██████████| 1326/1326 [01:06<00:00, 19.86it/s, loss=0.627]


Epoch 1: Train Acc=0.6401, Val Acc=0.8187, LR=0.001000
  -> Saved best model (Val Acc=0.8187)


Epoch 2/15: 100%|██████████| 1326/1326 [00:23<00:00, 55.61it/s, loss=0.425]


Epoch 2: Train Acc=0.8077, Val Acc=0.8484, LR=0.001000
  -> Saved best model (Val Acc=0.8484)


Epoch 3/15: 100%|██████████| 1326/1326 [00:25<00:00, 52.01it/s, loss=0.424]


...

Epoch 13: Train Acc=0.9450, Val Acc=0.9179, LR=0.000250


Epoch 14/15: 100%|██████████| 1326/1326 [00:38<00:00, 34.34it/s, loss=0.172] 


Epoch 14: Train Acc=0.9476, Val Acc=0.9171, LR=0.000250


Epoch 15/15: 100%|██████████| 1326/1326 [00:38<00:00, 34.52it/s, loss=0.227] 


Epoch 15: Train Acc=0.9494, Val Acc=0.9186, LR=0.000125
  -> Saved best model (Val Acc=0.9186)

Best Val Accuracy: 0.9186


In [10]:
model.load_state_dict(torch.load("best_model_gru.pth"))
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        pred = out.argmax(1)
        correct += (pred.cpu() == y).sum().item()
        total += y.size(0)

print(f"Test Accuracy: {correct/total:.4f}")

Test Accuracy: 0.9057


In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

model.load_state_dict(torch.load("best_model_gru.pth", map_location=device))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        pred = out.argmax(1).cpu().tolist()
        all_preds.extend(pred)
        all_labels.extend(y.tolist())

precision = precision_score(all_labels, all_preds, average='weighted')
recall    = recall_score(all_labels, all_preds, average='weighted')
f1        = f1_score(all_labels, all_preds, average='weighted')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=dataset.classes))

Precision: 0.9068
Recall:    0.9057
F1 Score:  0.9058

              precision    recall  f1-score   support

    backward       0.95      0.90      0.93       165
         bed       0.82      0.86      0.84       207
        bird       0.89      0.86      0.88       185
         cat       0.85      0.93      0.89       194
         dog       0.89      0.85      0.87       220
        down       0.91      0.88      0.89       406
       eight       0.93      0.92      0.92       408
        five       0.90      0.95      0.92       445
      follow       0.86      0.81      0.84       172
     forward       0.86      0.76      0.81       155
        four       0.87      0.89      0.88       400
          go       0.80      0.89      0.84       402
       happy       0.91      0.91      0.91       203
       house       0.96      0.92      0.94       191
       learn       0.72      0.81      0.76       161
        left       0.91      0.90      0.90       412
      marvin       0.96   

: 